# 09 — Grant App Service Principal Permissions

**Jackson and Jackson HR Digital**

Grants the Hire Right app service principal the Unity Catalog and Genie Space permissions it needs to serve data and run analytics.

### What this notebook does
1. Looks up the app SP from the deployed Databricks App
2. Grants `USE CATALOG`, `USE SCHEMA`, and `SELECT` on all tables in the schema to the SP
3. Grants `EXECUTE` on all UC functions in the schema to the SP
4. Grants `CAN_RUN` on the Genie Space to the SP via the REST API

**Prerequisites:** Notebook `07_deploy_app.ipynb` must have run (app SP exists)

## Setup

In [ ]:
%pip install -q databricks-sdk

In [ ]:
dbutils.library.restartPython()

In [ ]:
dbutils.widgets.text("catalog",              "bx4",                             "UC Catalog")
dbutils.widgets.text("schema",               "hrd_2030",                        "UC Schema")
dbutils.widgets.text("app_name",             "hire-right-agent-app",            "Databricks App Name")
dbutils.widgets.text("genie_space_id",       "01f13a0f6a081fabbea933cfb0db1d01", "Genie Space ID")
dbutils.widgets.text("warehouse_id",         "aa06644a4b0be047",                "SQL Warehouse ID")
dbutils.widgets.text("agent_endpoint_name",  "hire-right-agent-endpoint",       "Agent Endpoint Name")

In [ ]:
import os, requests
from databricks.sdk import WorkspaceClient

notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
project_root  = "/Workspace" + notebook_path.rsplit("/notebooks", 1)[0]
try:
    from dotenv import load_dotenv
    load_dotenv(f"{project_root}/.env")
except ImportError:
    pass

catalog              = dbutils.widgets.get("catalog")             or os.getenv("TARGET_CATALOG",          "bx4")
schema               = dbutils.widgets.get("schema")              or os.getenv("TARGET_SCHEMA",            "hrd_2030")
app_name             = dbutils.widgets.get("app_name")            or os.getenv("APP_NAME",                 "hire-right-agent-app")
genie_space_id       = dbutils.widgets.get("genie_space_id")      or os.getenv("GENIE_SPACE_ID",           "")
warehouse_id         = dbutils.widgets.get("warehouse_id")        or os.getenv("DATABRICKS_WAREHOUSE_ID",  "")
agent_endpoint_name  = dbutils.widgets.get("agent_endpoint_name") or os.getenv("DATABRICKS_AGENT_ENDPOINT","hire-right-agent-endpoint")

w_ctx   = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
host    = w_ctx.apiUrl().get().rstrip("/")
token   = w_ctx.apiToken().get()
headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

print(f"Catalog              : {catalog}")
print(f"Schema               : {schema}")
print(f"App Name             : {app_name}")
print(f"Genie Space ID       : {genie_space_id or '(not set)'}")
print(f"Warehouse ID         : {warehouse_id or '(not set)'}")
print(f"Agent Endpoint Name  : {agent_endpoint_name}")

## Resolve App Service Principal

In [ ]:
resp = requests.get(
    f"{host}/api/2.0/apps/{app_name}",
    headers=headers,
    timeout=30,
)
resp.raise_for_status()
app_info     = resp.json()
sp_id        = app_info.get("service_principal_id")
sp_client_id = app_info.get("service_principal_client_id")

if not sp_client_id:
    raise ValueError(f"App '{app_name}' has no service principal — has it been deployed yet?")

print(f"App SP numeric ID : {sp_id}")
print(f"App SP client ID  : {sp_client_id}")

## Grant Unity Catalog Permissions

Grant the SP access to the catalog, schema, all tables, and all UC functions.

In [ ]:
grants = [
    f"GRANT USE CATALOG ON CATALOG {catalog} TO `{sp_client_id}`",
    f"GRANT USE SCHEMA ON SCHEMA {catalog}.{schema} TO `{sp_client_id}`",
    f"GRANT SELECT ON SCHEMA {catalog}.{schema} TO `{sp_client_id}`",
    f"GRANT EXECUTE ON SCHEMA {catalog}.{schema} TO `{sp_client_id}`",
    f"GRANT READ VOLUME ON VOLUME {catalog}.{schema}.raw_data TO `{sp_client_id}`",
]

for sql in grants:
    try:
        spark.sql(sql)
        print(f"\u2713 {sql}")
    except Exception as e:
        print(f"\u26a0\ufe0f  {sql}\n    {e}")

print("\n\u2713 Unity Catalog grants complete")

## Grant Genie Space CAN_RUN

Grant the app SP `CAN_RUN` on the Genie Space via the REST permissions API so it can execute queries on behalf of the app.

In [ ]:
if not genie_space_id:
    print("\u26a0\ufe0f  genie_space_id not set — skipping Genie permission grant.")
else:
    payload = {
        "access_control_list": [
            {
                "service_principal_name": str(sp_client_id),
                "permission_level": "CAN_RUN"
            }
        ]
    }

    resp = requests.patch(
        f"{host}/api/2.0/permissions/genie/{genie_space_id}",
        headers=headers,
        json=payload,
        timeout=30,
    )

    if resp.status_code in (200, 204):
        print(f"\u2713 Granted CAN_RUN on Genie Space {genie_space_id} to SP {sp_client_id}")
    else:
        print(f"\u26a0\ufe0f  Genie grant returned HTTP {resp.status_code}: {resp.text[:300]}")

In [ ]:
if not warehouse_id:
    print("\u26a0\ufe0f  warehouse_id not set — skipping SQL Warehouse permission grant.")
else:
    payload = {
        "access_control_list": [
            {
                "service_principal_name": str(sp_client_id),
                "permission_level": "CAN_USE"
            }
        ]
    }

    resp = requests.patch(
        f"{host}/api/2.0/permissions/warehouses/{warehouse_id}",
        headers=headers,
        json=payload,
        timeout=30,
    )

    if resp.status_code in (200, 204):
        print(f"\u2713 Granted CAN_USE on SQL Warehouse {warehouse_id} to SP {sp_client_id}")
    else:
        print(f"\u26a0\ufe0f  Warehouse grant returned HTTP {resp.status_code}: {resp.text[:300]}")

In [ ]:
# Grant app SP CAN_QUERY on the agent serving endpoint.
# The endpoint is deleted and recreated on every agent deployment (notebook 06),
# so permissions are wiped each run and must be re-granted here.
if not agent_endpoint_name:
    print("⚠️  agent_endpoint_name not set — skipping agent endpoint permission grant.")
else:
    # Resolve endpoint ID from name
    ep_resp = requests.get(
        f"{host}/api/2.0/serving-endpoints/{agent_endpoint_name}",
        headers=headers,
        timeout=30,
    )
    if not ep_resp.ok:
        print(f"⚠️  Could not look up endpoint '{agent_endpoint_name}': HTTP {ep_resp.status_code}: {ep_resp.text[:200]}")
    else:
        ep_id = ep_resp.json().get("id")
        payload = {
            "access_control_list": [
                {
                    "service_principal_name": str(sp_client_id),
                    "permission_level": "CAN_QUERY"
                }
            ]
        }
        perm_resp = requests.patch(
            f"{host}/api/2.0/permissions/serving-endpoints/{ep_id}",
            headers=headers,
            json=payload,
            timeout=30,
        )
        if perm_resp.status_code in (200, 204):
            print(f"✓ Granted CAN_QUERY on agent endpoint '{agent_endpoint_name}' (id={ep_id}) to SP {sp_client_id}")
        else:
            print(f"⚠️  Agent endpoint grant returned HTTP {perm_resp.status_code}: {perm_resp.text[:300]}")

## Summary

| Grant | Target |
|---|---|
| `USE CATALOG` on `bx4` | App SP |
| `USE SCHEMA` on `bx4.hrd_2030` | App SP |
| `SELECT` on all tables in `bx4.hrd_2030` | App SP |
| `EXECUTE` on all functions in `bx4.hrd_2030` | App SP |
| `CAN_RUN` on Genie Space | App SP |